## Outline

- DF for minute features at date grain (Done)
- DF for daily features at date grain (Done)
- DF for returns over 1, 3 and 5 days (Done)
- Simple logistic, rfc and xgb models for daily alone, min alone and then combined
- Permutation importance
- Chart over rolling 5 days for 25 iterations, aka 6 months

In [ ]:
import min_features, daily_return
import importlib
import pandas as pd

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 3, 5]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = df_daily.iloc[:, 1:].columns.difference(return_cols).to_list()
min_cols = df_min.iloc[:, 1:].columns.to_list()

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score

column_sets = [daily_cols]#, min_cols, daily_cols + min_cols]
names = ['daily']#, 'minute', 'daily+minute']
returns = [1]#, 3, 5]
runs = 5
test_size = 5
lbs = [6]
offset_size = test_size

models = {
    "logistic": LogisticRegression(max_iter=1000),
    "linear_svm": LinearSVC(),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

tscv = TimeSeriesSplit(n_splits=5)

results = []

for feature_cols, name in zip(column_sets, names):

    X = df_main[feature_cols].to_numpy()

    for r in returns:
        
        y = df_main[f"Return_{r}"].to_numpy()
        
        for i in range(runs): # number of runs to do

            offset = max(i * offset_size, 0) # step size for each run
            
            for lb in lbs:
                 
                df_ph = df_main.iloc[offset : offset + 245 * lb, :].copy()  # 245 records is ~1 year of data
                ret_col = f"Return_{r}"
                ret_pct_col = f"Return%_{r}"

                rets = df_ph[ret_pct_col]
                neg, pos = rets[rets < 0], rets[rets > 0]

                neg_cut = neg.nlargest(max(1, int(len(neg) * 0.05))).min()
                pos_cut = pos.nsmallest(max(1, int(len(pos) * 0.05))).max()
                filtered = df_ph[(rets < neg_cut) | (rets > pos_cut)].copy()
                print(f"Run {i+1} of {runs} | LB: {lb} | Horizon: {r} | {filtered['Date'].iloc[test_size]} - {filtered['Date'].iloc[0]}")

                df_indicators = filtered[feature_cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = filtered[ret_col]


Run 1 of 5 | LB: 6 | Horizon: 1 | 2025-12-12 - 2025-12-19
Run 2 of 5 | LB: 6 | Horizon: 1 | 2025-12-04 - 2025-12-12
Run 3 of 5 | LB: 6 | Horizon: 1 | 2025-11-25 - 2025-12-05
Run 4 of 5 | LB: 6 | Horizon: 1 | 2025-11-19 - 2025-11-26
Run 5 of 5 | LB: 6 | Horizon: 1 | 2025-11-10 - 2025-11-19


In [ ]:
def eval_models_timeseries_cv(df_window, feature_cols, target_col, n_splits=5):
    X = df_window[feature_cols].to_numpy()
    y = df_window[target_col].to_numpy()

    # normalize {-1,1} -> {0,1} if needed
    u = np.unique(y[~pd.isna(y)])
    if set(u.tolist()) == {0, 1}:
        y = (y > 0).astype(int)

    tscv = TimeSeriesSplit(n_splits=n_splits)

    rows = []
    for model_name, model in models.items():
        bal_accs, f1s = [], []

        # validation-only distribution trackers
        val_ns, val_pos_fracs, val_pos_ns, val_neg_ns = [], [], [], []

        for tr_idx, va_idx in tscv.split(X):
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model.fit(X_tr, y_tr)
            preds = model.predict(X_va)

            bal_accs.append(balanced_accuracy_score(y_va, preds))
            f1s.append(f1_score(y_va, preds))

            n_va = len(y_va)
            n_pos = int((y_va == 1).sum())
            n_neg = int((y_va == 0).sum())

            val_ns.append(n_va)
            val_pos_ns.append(n_pos)
            val_neg_ns.append(n_neg)
            val_pos_fracs.append(n_pos / n_va if n_va else np.nan)

        rows.append({
            "model": model_name,
            "bal_acc_mean": float(np.mean(bal_accs)),
            "bal_acc_std": float(np.std(bal_accs)),
            "f1_mean": float(np.mean(f1s)),
            "f1_std": float(np.std(f1s)),

            # validation-only distribution summary across folds
            "val_n_mean": float(np.mean(val_ns)),
            "val_pos_frac_mean": float(np.nanmean(val_pos_fracs)),
            "val_pos_n_mean": float(np.mean(val_pos_ns)),
            "val_neg_n_mean": float(np.mean(val_neg_ns)),
            "val_pos_frac_min": float(np.nanmin(val_pos_fracs)),
            "val_pos_frac_max": float(np.nanmax(val_pos_fracs)),
        })

    return pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False)

column_sets = [daily_cols]#, min_cols, daily_cols + min_cols]
names = ['daily']#, 'minute', 'daily+minute']
returns = [1]#, 3, 5]
runs = 6
test_size = 5
lbs = [6]
offset_size = test_size

models = {
    "logistic": LogisticRegression(max_iter=1000),
    "linear_svm": LinearSVC(),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "naive_bayes": GaussianNB(),
}

tscv = TimeSeriesSplit(n_splits=5)

results = []
results_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        ret_col = f"Return_{r}"
        ret_pct_col = f"Return%_{r}"

        for i in range(runs):
            offset = max(i * offset_size, 0)

            for lb in lbs:
                df_ph = df_main.iloc[offset : offset + 245 * lb, :].copy()

                rets = df_ph[ret_pct_col]
                neg, pos = rets[rets < 0], rets[rets > 0]
                neg_cut = neg.nlargest(max(1, int(len(neg) * 0.05))).min()
                pos_cut = pos.nsmallest(max(1, int(len(pos) * 0.05))).max()

                filtered = df_ph[(rets < neg_cut) | (rets > pos_cut)].copy()

                # Basic cleaning consistent with your pipeline
                filtered[feature_cols] = filtered[feature_cols].replace([np.inf, -np.inf], 0)

                print(
                    f"Run {i+1}/{runs} | Feats:{feat_name} | LB:{lb} | H:{r} | "
                    f"{filtered['Date'].iloc[test_size]} - {filtered['Date'].iloc[0]}"
                )

                fold_scores = eval_models_timeseries_cv(
                    df_window=filtered,
                    feature_cols=feature_cols,
                    target_col=ret_col,
                    n_splits=5,
                )

                # attach metadata so you can compare across pipeline dimensions
                fold_scores["feature_set"] = feat_name
                fold_scores["horizon"] = r
                fold_scores["run"] = i
                fold_scores["lb_years"] = lb
                fold_scores["start_date"] = filtered["Date"].iloc[0]
                fold_scores["end_date"] = filtered["Date"].iloc[-1]
                fold_scores["n_rows"] = len(filtered)

                results_all.append(fold_scores)

results_df = pd.concat(results_all, ignore_index=True)

Run 1/5 | Feats:daily | LB:6 | H:1 | 2025-12-12 - 2025-12-19


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 2/5 | Feats:daily | LB:6 | H:1 | 2025-12-04 - 2025-12-12


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 3/5 | Feats:daily | LB:6 | H:1 | 2025-11-25 - 2025-12-05


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 4/5 | Feats:daily | LB:6 | H:1 | 2025-11-19 - 2025-11-26


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

Run 5/5 | Feats:daily | LB:6 | H:1 | 2025-11-10 - 2025-11-19


/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/brettchase/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

In [12]:
results_df

,model,bal_acc_mean,bal_acc_std,f1_mean,f1_std,val_n_mean,val_pos_frac_mean,val_pos_n_mean,val_neg_n_mean,val_pos_frac_min,val_pos_frac_max,feature_set,horizon,run,lb_years,start_date,end_date,n_rows
0,random_forest,1.000000,0.000000,1.000000,0.000000,215.0,0.55814,120.0,95.0,0.446512,0.627907,daily,1,0,6,2025-12-19,2020-01-31,1292
1,grad_boost,1.000000,0.000000,1.000000,0.000000,215.0,0.55814,120.0,95.0,0.446512,0.627907,daily,1,0,6,2025-12-19,2020-01-31,1292
2,logistic,0.540089,0.026680,0.642194,0.118512,215.0,0.55814,120.0,95.0,0.446512,0.627907,daily,1,0,6,2025-12-19,2020-01-31,1292
3,linear_svm,0.500000,0.000000,0.714321,0.052864,215.0,0.55814,120.0,95.0,0.446512,0.627907,daily,1,0,6,2025-12-19,2020-01-31,1292
4,naive_bayes,0.491634,0.027346,0.670413,0.053557,215.0,0.55814,120.0,95.0,0.446512,0.627907,daily,1,0,6,2025-12-19,2020-01-31,1292
5,random_forest,1.000000,0.000000,1.000000,0.000000,215.0,0.55814,120.0,95.0,0.441860,0.627907,daily,1,1,6,2025-12-12,2020-01-24,1292
6,grad_boost,1.000000,0.000000,1.000000,0.000000,215.0,0.55814,120.0,95.0,0.441860,0.627907,daily,1,1,6,2025-12-12,2020-01-24,1292
7,logistic,0.520790,0.028506,0.658080,0.077665,215.0,0.55814,120.0,95.0,0.441860,0.627907,daily,1,1,6,2025-12-12,2020-01-24,1292
8,linear_svm,0.500000,0.000000,0.714117,0.055432,215.0,0.55814,120.0,95.0,0.441860,0.627907,daily,1,1,6,2025-12-12,2020-01-24,1292
9,naive_bayes,0.492174,0.026747,0.678442,0.056995,215.0,0.55814,120.0,95.0,0.441860,0.627907,daily,1,1,6,2025-12-12,2020-01-24,1292
